In [1]:
from scipy.sparse import kron, eye
from scipy.sparse import diags
import numpy as np

In [2]:
dim = 5
n = 8

# A

In [ ]:
# create stiffness matrix for 1d: L
# embed it into higher dim with kron(I, L)
# in each step: add kron(L_1d, I), where L_1d in of size (N,N)
main_diag = 2 * np.ones(n, dtype=int)
off_diag = -1 * np.ones(n-1, dtype=int)
L = diags([off_diag, main_diag, off_diag], offsets=[-1, 0, 1], shape=(n, n), dtype=int)
I = eye(n, dtype=int)
A = L
for d in range(1,dim):
    A = kron(I, A) + kron(L, eye(n**d, dtype=int))
A = A.toarray()
print(A) # This is the 2D Poisson stiffness matrix

[[10 -1  0 ...  0  0  0]
 [-1 10 -1 ...  0  0  0]
 [ 0 -1 10 ...  0  0  0]
 ...
 [ 0  0  0 ... 10 -1  0]
 [ 0  0  0 ... -1 10 -1]
 [ 0  0  0 ...  0 -1 10]]


In [4]:
A.shape

(32768, 32768)

# B

In [5]:
def to_nd_from_1d(i):
    k = np.zeros([dim], dtype=int)
    i_rem = i
    for j in range(dim):
        kj = i_rem // n**(dim-(j+1))
        i_rem -= kj * n**(dim-(j+1))
        k[j] = kj
    return k

In [6]:
to_nd_from_1d(9)

array([0, 0, 0, 1, 1])

In [7]:
def to_1d_from_nd(k):
    return np.dot(k, n**np.arange(dim-1,-1,-1))

In [8]:
to_1d_from_nd(np.array([0,0,0,1,1]))

np.int64(9)

In [9]:
N = n**dim
indx_min = 0
indx_max = n-1
B = np.zeros((N,N), dtype=int)
for i in range(N): # go over each row of the stiffness matrix
    k = to_nd_from_1d(i)
    B[i,i] = 2*dim
    for d in range(dim):
        if k[d]+1 <= indx_max:
            B[i,i+n**(dim-(d+1))] = -1
        if k[d]-1 >= indx_min:
            B[i,i-n**(dim-(d+1))] = -1
B

array([[10, -1,  0, ...,  0,  0,  0],
       [-1, 10, -1, ...,  0,  0,  0],
       [ 0, -1, 10, ...,  0,  0,  0],
       ...,
       [ 0,  0,  0, ..., 10, -1,  0],
       [ 0,  0,  0, ..., -1, 10, -1],
       [ 0,  0,  0, ...,  0, -1, 10]], shape=(32768, 32768))

In [10]:
B.shape

(32768, 32768)

# compare A & B

In [11]:
np.nonzero(A - B)

(array([], dtype=int64), array([], dtype=int64))